# Week 4: LangChain Fundamentals

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week04-langchain.html

**How to use:** run the cells from the top (Runtime → Run all, or Shift+Enter cell by cell). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need a key: add it with the 🔑 Secrets panel on the left, or set `os.environ["OPENAI_API_KEY"] = "..."` in a cell. Examples that use Ollama need a local Ollama server and will not work in Colab. For GPU examples choose Runtime → Change runtime type → T4 GPU.

In [ ]:
# Libraries this lesson uses that are not preinstalled in Colab
%pip install -q langchain-community langchain-core langchain-openai langgraph

## Setting Up the Environment

In [ ]:
# Install required packages (run in a terminal):
#   pip install langchain langchain-community langchain-openai langgraph sqlalchemy requests

# Import necessary modules
import os
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool
from langgraph.prebuilt import create_react_agent

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

## Database Connection

In [ ]:
# Connect to an SQLite Database
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

# For an existing SQLite database
db_path = "chinook.db"  # Path to your SQLite database
engine = create_engine(f"sqlite:///{db_path}")
db = SQLDatabase(engine)

# Alternatively, for in-memory testing with example data
def get_in_memory_db():
    import sqlite3
    import requests

    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)

    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )

engine = get_in_memory_db()
db = SQLDatabase(engine)

## Creating the LangChain Agent

In [ ]:
# Initialize Language Model
llm = ChatOpenAI(model="gpt-4o", temperature=0)  # You can also use gpt-4o-mini for lower cost

# Initialize SQL Database Toolkit
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Get the tools from the toolkit
tools = toolkit.get_tools()

# Build the system prompt from LangChain's standard SQL-agent prompt
# (the same text as the "langchain-ai/sql-agent-system-prompt" prompt on LangChain Hub)
system_message = SQL_PREFIX.format(dialect="SQLite", top_k=5)

# Create the agent
agent_executor = create_react_agent(
    llm,
    tools,
    prompt=system_message
)

## Django Integration (views.py)

This code is a file of an application, not a notebook step. The cell saves it as `views.py`; run it from a terminal on your computer (or with `!python views.py` here, if it is a script).

In [ ]:
%%writefile views.py
# views.py for Django integration
from django.http import JsonResponse
from django.views.decorators.csrf import csrf_exempt
import json

@csrf_exempt
def chat_with_db(request):
    if request.method == 'POST':
        try:
            data = json.loads(request.body)
            user_message = data.get('message', '')

            # Process the user message with our agent
            response = agent_executor.invoke(
                {"messages": [("user", user_message)]}
            )

            # Extract the final AI message
            ai_message = response["messages"][-1].content

            return JsonResponse({
                'success': True,
                'message': ai_message
            })

        except Exception as e:
            return JsonResponse({
                'success': False,
                'error': str(e)
            })

    return JsonResponse({
        'success': False,
        'error': 'Only POST requests are supported'
    })

## Run in your browser

**Retrieval for RAG: embed chunks, rank by cosine similarity, build the prompt**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks = [
    "Transformers use self-attention to relate all tokens in a sequence.",
    "BERT is an encoder-only model pre-trained with masked language modelling.",
    "GPT models are decoder-only and generate text left to right.",
    "Amman is the capital of Jordan and its largest city.",
    "Petra is a historical city in southern Jordan carved into rock.",
    "A vector store returns the chunks whose embeddings are nearest to the query.",
]
question = "How do GPT and BERT models differ?"
k = 2

# TF-IDF stands in for an embedding model here: it matches shared words, whereas a real
# embedding model (e.g. all-MiniLM-L6-v2) also matches meaning, such as "capital" ~ "largest city".
vec = TfidfVectorizer().fit(chunks + [question])
sims = cosine_similarity(vec.transform([question]), vec.transform(chunks))[0]
top = sims.argsort()[::-1][:k]

print("ranking:")
for i in sims.argsort()[::-1]:
    print(f"  {sims[i]:.3f}  {chunks[i]}")

context = "\n".join(f"- {chunks[i]}" for i in top)
prompt = f"Answer using only the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
print("\nprompt sent to the LLM:\n" + prompt)

**A tiny "chain": prompt template -> model -> output parser (pure Python)**

In [ ]:
def prompt_template(topic):
    return f"List three key facts about {topic}, one per line."

def fake_llm(prompt):                 # stands in for ChatOpenAI / Ollama
    facts = {"MCP": ["Uses JSON-RPC 2.0", "Servers expose tools, resources and prompts", "Transports: stdio and Streamable HTTP"]}
    topic = prompt.split("about ")[1].split(",")[0]
    return "\n".join(facts.get(topic, ["(no facts)"]))

def output_parser(text):
    return [line.strip() for line in text.splitlines() if line.strip()]

def chain(topic):                     # LCEL equivalent: prompt | llm | parser
    return output_parser(fake_llm(prompt_template(topic)))

print(chain("MCP"))

## Test code for the course environment

**Verify installation and call a chat model directly**

In [ ]:
# 1. Verify installation and call a chat model directly
# pip install langchain langchain-core langchain-openai
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

os.environ["OPENAI_API_KEY"] = "your-api-key-here"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(llm.invoke([HumanMessage(content="In one sentence, what is LangChain?")]).content)

**A minimal LCEL chain: prompt template -> LLM -> output parser**

In [ ]:
# 2. A minimal LCEL chain: prompt template -> LLM -> output parser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise tutor for a data-science course."),
    ("human", "Explain {topic} in two sentences."),
])
chain = prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

print(chain.invoke({"topic": "LangChain Expression Language"}))

**Inspect the pieces of a chain without an API key**

In [ ]:
# 3. Inspect the pieces of a chain without an API key
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Summarize {text} for a {audience}."),
])
messages = prompt.format_messages(text="the LangChain stack", audience="first-year student")
for m in messages:
    print(f"{m.type}: {m.content}")
print("Input variables:", prompt.input_variables)

**Stream tokens from a chain (streaming is first-class in LCEL)**

In [ ]:
# 4. Stream tokens from a chain (streaming is first-class in LCEL)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

chain = (
    ChatPromptTemplate.from_template("List three uses of {tool} as bullet points.")
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)
for chunk in chain.stream({"tool": "LangSmith"}):
    print(chunk, end="", flush=True)
print()

**Load Chinook into an in-memory SQLite DB and inspect it with SQLDatabase (no LLM needed)**

In [ ]:
# 1. Load Chinook into an in-memory SQLite DB and inspect it with SQLDatabase (no LLM needed)
# pip install langchain-community sqlalchemy requests
import sqlite3, requests
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool
from langchain_community.utilities import SQLDatabase

url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.executescript(requests.get(url).text)
engine = create_engine("sqlite://", creator=lambda: conn, poolclass=StaticPool,
                       connect_args={"check_same_thread": False})
db = SQLDatabase(engine)

print(db.dialect)
print(db.get_usable_table_names())
print(db.get_table_info(["Customer", "Invoice"])[:800])

**Run the Process Flow SQL by hand and confirm the "USA, 523.06" result**

In [ ]:
# 2. Run the Process Flow SQL by hand and confirm the "USA, 523.06" result
sql = """
SELECT c.Country, SUM(i.Total) AS TotalSpent
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.Country
ORDER BY TotalSpent DESC
LIMIT 1;
"""
print(db.run(sql))

**Look at the four tools the SQLDatabaseToolkit exposes**

In [ ]:
# 3. Look at the four tools the SQLDatabaseToolkit exposes
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
for t in toolkit.get_tools():
    print(f"{t.name}: {t.description[:90]}...")

**Ask the SQL agent a question and watch every intermediate step**

In [ ]:
# 4. Ask the SQL agent a question and watch every intermediate step
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX
from langgraph.prebuilt import create_react_agent

system_message = SQL_PREFIX.format(dialect="SQLite", top_k=5)
agent_executor = create_react_agent(llm, toolkit.get_tools(), prompt=system_message)

for step in agent_executor.stream(
    {"messages": [("user", "Who are the top 3 best selling artists?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()